# CSCI316 Project 2 — Hugging Face Transformers
**Task:** Sentiment Analysis on Tamil-English Code-Switched Text

This notebook mirrors the same pre-PEFT implementation flow, but is focused specifically on a Hugging Face Transformers workflow.

## Cell 1 — Install Dependencies

In [ ]:
!pip install transformers datasets evaluate accelerate sentencepiece pandas scikit-learn matplotlib seaborn --quiet

## Cell 2 — Imports & Reproducibility

In [ ]:
import os
import json
import random
import numpy as np
import pandas as pd

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)

from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

## Cell 3 — Configuration

In [ ]:
CONFIG = {
    "train_path": "tamilmix_train.csv",
    "val_path": "tamilmix_val.csv",
    "test_path": "tamilmix_test.csv",
    "model_name": "google/mt5-small",
    "num_labels": 5,
    "max_length": 128,
    "batch_size": 16,
    "num_epochs": 5,
    "learning_rate": 2e-5,
    "weight_decay": 0.01,
    "warmup_ratio": 0.1,
    "output_dir": "hf_mt5_model",
    "results_dir": "results_hf_transformers",
}

LABEL_NAMES = ["Positive", "Negative", "Mixed_feelings", "unknown_state", "not-Tamil"]

os.makedirs(CONFIG["output_dir"], exist_ok=True)
os.makedirs(CONFIG["results_dir"], exist_ok=True)
print("Config loaded.")
print(f"Model: {CONFIG['model_name']}")

## Cell 4 — Load Cleaned Data

In [ ]:
train_df = pd.read_csv(CONFIG["train_path"])
val_df = pd.read_csv(CONFIG["val_path"])
test_df = pd.read_csv(CONFIG["test_path"])

for df in [train_df, val_df, test_df]:
    df["text_clean"] = df["text_clean"].fillna("").astype(str)
    df["label_int"] = df["label"].astype(int)

print(f"Train:      {len(train_df)} rows")
print(f"Validation: {len(val_df)} rows")
print(f"Test:       {len(test_df)} rows")
print("\nLabel distribution (train):")
print(train_df["label_int"].value_counts().sort_index())

## Cell 5 — Build Hugging Face Datasets

In [ ]:
keep_cols = ["text_clean", "label_int"]

hf_train = Dataset.from_pandas(train_df[keep_cols].rename(columns={"label_int": "labels"}), preserve_index=False)
hf_val = Dataset.from_pandas(val_df[keep_cols].rename(columns={"label_int": "labels"}), preserve_index=False)
hf_test = Dataset.from_pandas(test_df[keep_cols].rename(columns={"label_int": "labels"}), preserve_index=False)

print(hf_train)
print(hf_val)
print(hf_test)

## Cell 6 — Tokenizer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_name"], use_fast=False)

def tokenize_batch(batch):
    return tokenizer(
        batch["text_clean"],
        truncation=True,
        max_length=CONFIG["max_length"],
    )

hf_train_tok = hf_train.map(tokenize_batch, batched=True)
hf_val_tok = hf_val.map(tokenize_batch, batched=True)
hf_test_tok = hf_test.map(tokenize_batch, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

sample = "thala vera level . luv u so much"
print(tokenizer(sample))

## Cell 7 — Model Initialisation

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    CONFIG["model_name"],
    num_labels=CONFIG["num_labels"],
)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

## Cell 8 — Metrics Function

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_weighted": f1_score(labels, preds, average="weighted", zero_division=0),
    }

## Cell 9 — Training Arguments

In [ ]:
num_train_steps = (len(hf_train_tok) // CONFIG["batch_size"]) * CONFIG["num_epochs"]
warmup_steps = int(num_train_steps * CONFIG["warmup_ratio"])

training_args = TrainingArguments(
    output_dir=CONFIG["output_dir"],
    num_train_epochs=CONFIG["num_epochs"],
    per_device_train_batch_size=CONFIG["batch_size"],
    per_device_eval_batch_size=CONFIG["batch_size"],
    learning_rate=CONFIG["learning_rate"],
    weight_decay=CONFIG["weight_decay"],
    warmup_steps=warmup_steps,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="f1_weighted",
    greater_is_better=True,
    save_total_limit=2,
    report_to="none",
    seed=SEED,
)

print(f"Train steps: {num_train_steps}")
print(f"Warmup steps: {warmup_steps}")

## Cell 10 — Trainer Setup

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=hf_train_tok,
    eval_dataset=hf_val_tok,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

## Cell 11 — Train

In [ ]:
train_result = trainer.train()
train_metrics = train_result.metrics
print(train_metrics)

## Cell 12 — Evaluate on Validation and Test

In [ ]:
val_metrics = trainer.evaluate(eval_dataset=hf_val_tok)
test_metrics = trainer.evaluate(eval_dataset=hf_test_tok)

print("Validation metrics:")
print(val_metrics)
print("\nTest metrics:")
print(test_metrics)

## Cell 13 — Classification Report + Confusion Matrix

In [ ]:
pred_output = trainer.predict(hf_test_tok)
test_logits = pred_output.predictions
test_labels = pred_output.label_ids
test_preds = np.argmax(test_logits, axis=-1)

print(classification_report(test_labels, test_preds, target_names=LABEL_NAMES, digits=4, zero_division=0))

cm = confusion_matrix(test_labels, test_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES)
plt.title("Confusion Matrix (Test)")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.tight_layout()
plt.show()

## Cell 14 — Inference Helper

In [ ]:
def predict_text(text: str):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=CONFIG["max_length"])
    outputs = trainer.model(**inputs)
    probs = outputs.logits.softmax(dim=-1).detach().cpu().numpy()[0]
    pred_id = int(np.argmax(probs))
    return {
        "label_id": pred_id,
        "label_name": LABEL_NAMES[pred_id],
        "probabilities": {LABEL_NAMES[i]: float(probs[i]) for i in range(len(LABEL_NAMES))},
    }

print(predict_text("This movie was super, but konjam long-a irundhudhu"))

## Cell 15 — Save Model and Metrics

In [ ]:
trainer.save_model(CONFIG["output_dir"])
tokenizer.save_pretrained(CONFIG["output_dir"])

metrics = {
    "train": train_metrics,
    "validation": val_metrics,
    "test": test_metrics,
}

metrics_path = os.path.join(CONFIG["results_dir"], "metrics_hf_transformers.json")
with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)

print(f"Saved model to: {CONFIG['output_dir']}")
print(f"Saved metrics to: {metrics_path}")